In [ ]:
import numpy as np
import pandas as pd

#importation fichier .parquet
httrain= pd.read_parquet("history_toulouse_train.parquet").copy()
httest=pd.read_parquet("history_toulouse_test.parquet").copy()
httest

In [ ]:
#ajout serie(colonne) heure en fomrat datetime
httrain["heure"]=pd.to_datetime(httrain.index.get_level_values("commit_at")).hour
httrain["minute"]=pd.to_datetime(httrain.index.get_level_values("commit_at")).minute

httrain

In [ ]:
#calcul moyenne des velo par quart d'heure
meanbh = httrain.groupby(["heure", "minute"])["bikes"].mean()
     
     
meanbh   


In [ ]:
# ajout des resultat dans httest
target_columns = [
    "bikes_+15min", "bikes_+30min", "bikes_+45min", "bikes_+60min",
    "bikes_+75min", "bikes_+90min", "bikes_+105min", "bikes_+120min"
]

delta_hour = [15, 30, 45, 60, 75, 90, 105, 120]


forecast= pd.to_datetime(httest.index.get_level_values("commit_at"))

for columns in range(8):
    
    time_delta= forecast + pd.Timedelta(minutes=delta_hour[columns])

    
    cles_recherche = pd.Series(list(zip(time_delta.hour, time_delta.minute)), index=httest.index)

    moyennes_futures = cles_recherche.map(meanbh)

    httest[target_columns[columns]] =  httest[target_columns[columns]].fillna(moyennes_futures).round().astype(int)


httest

In [ ]:
hour_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().hour
print(hour_forecast)
minute_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().minute
print(minute_forecast)